In [3]:
import requests
from getpass import getpass

serviceUrl = "https://m2m.cr.usgs.gov/api/api/json/stable/"
def prompt_ERS_login(serviceURL):
    print("Logging in...\n")

    p = ['Enter EROS Registration System (ERS) Username: ', 'Enter ERS Account Token: ']

    # Use requests.post() to make the login request
    response = requests.post(f"{serviceUrl}login-token", json={'username': getpass(prompt=p[0]), 'token': getpass(prompt=p[1])})

    if response.status_code == 200:  # Check for successful response
        apiKey = response.json()['data']
        print('\nLogin Successful, API Key Received!')
        headers = {'X-Auth-Token': apiKey}
        return apiKey
    else:
        print("\nLogin was unsuccessful, please try again or create an account at: https://ers.cr.usgs.gov/register.")

apiKey = prompt_ERS_login(serviceUrl)

Logging in...


Login Successful, API Key Received!


In [4]:
import json
import sys
import requests
# Send HTTP request
def sendRequest(url, data, apiKey=None, exitIfNoResponse=True):
    """
    Send a request to an M2M (Machine-to-Machine) endpoint and return the parsed JSON response.

    Parameters:
    - url (str): The URL of the M2M endpoint.
    - data (dict): The payload to be sent with the request.
    - apiKey (str, optional): An optional API key for authorization. If not provided, the request will be sent without an authorization header.
    - exitIfNoResponse (bool, optional): If True, the program will exit upon receiving an error or no response. Defaults to True.

    Returns:
    - dict: The parsed JSON response containing the data, or False if there was an error.
    """
    
    # Convert payload to json string
    json_data = json.dumps(data)
    
    if apiKey == None:
        response = requests.post(url, json_data)
    else:
        headers = {'X-Auth-Token': apiKey}              
        response = requests.post(url, json_data, headers = headers)  
    
    try:
      httpStatusCode = response.status_code 
      if response == None:
          print("No output from service")
          if exitIfNoResponse: sys.exit()
          else: return False
      output = json.loads(response.text)
      if output['errorCode'] != None:
          print(output['errorCode'], "- ", output['errorMessage'])
          if exitIfNoResponse: sys.exit()
          else: return False
      if  httpStatusCode == 404:
          print("404 Not Found")
          if exitIfNoResponse: sys.exit()
          else: return False
      elif httpStatusCode == 401: 
          print("401 Unauthorized")
          if exitIfNoResponse: sys.exit()
          else: return False
      elif httpStatusCode == 400:
          print("Error Code", httpStatusCode)
          if exitIfNoResponse: sys.exit()
          else: return False
    except Exception as e: 
          response.close()
          print(e)
          if exitIfNoResponse: sys.exit()
          else: return False
    response.close()
    
    return output['data']

username = "Pheobe"
token = "rC9ZtvWS9A7zSrymMsuI6rQVm2uSGyXYVlapcL08VPMLGSyR1xzg95a6W88gKlyr"
print("Logging in...\n")
    
serviceUrl = "https://m2m.cr.usgs.gov/api/api/json/stable/"
login_payload = {'username' : username, 'token' : token}
    
apiKey = sendRequest(serviceUrl + "login-token", login_payload)
    
print("API Key: " + apiKey + "\n")

Logging in...

API Key: eyJjaWQiOjI3NzgyNDk1LCJzIjoiMTc2NTIxNTY0MSIsInIiOjE0OCwicCI6WyJ1c2VyIiwiZG93bmxvYWQiLCJvcmRlciJdfQ==



In [7]:
import json
import requests
import datetime
import time
from pathlib import Path

# ========= 配置区：改这里 =========
USERNAME = "Pheobe"                 # 你的 USGS 账号
TOKEN    = "rC9ZtvWS9A7zSrymMsuI6rQVm2uSGyXYVlapcL08VPMLGSyR1xzg95a6W88gKlyr"        # ⚠️ 这里填你真正的 M2M token（不要当成 password 用）

# AOI：最小纬度/经度，最大纬度/经度（WGS84）
LAT_MIN, LON_MIN = 31.0, -105.5
LAT_MAX, LON_MAX = 32.0, -104.5

# 时间范围（字符串格式：YYYY-MM-DD）
START_DATE = "2022-06-01"
END_DATE   = "2022-06-10"

OUTPUT_DIR = Path("./landsat_downloads")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Landsat 8/9 Collection 2 Level-2 数据集名称
DATASET_NAME = "landsat_ot_c2_l2"
SERVICE_URL = "https://m2m.cr.usgs.gov/api/api/json/stable/"

# ========= 通用请求函数（官方示例同款结构，去掉默认 api_key） =========
def send_request(url, data, api_key=None, exit_if_error=True):
    """
    对 USGS M2M 发送 POST 请求:
    - api_key 为 None 时，不带 X-Auth-Token（用于 login / login-token）
    - api_key 不为 None 时，在 header 中带上 X-Auth-Token
    """
    payload = json.dumps(data)

    if api_key is None:
        resp = requests.post(url, payload)
    else:
        headers = {"X-Auth-Token": api_key}
        resp = requests.post(url, payload, headers=headers)

    try:
        http_status = resp.status_code
        out = json.loads(resp.text)

        # M2M 逻辑错误
        if out.get("errorCode") is not None:
            print(out["errorCode"], "-", out.get("errorMessage"))
            if exit_if_error:
                raise RuntimeError("M2M error")
            return None

        # HTTP 层错误
        if http_status in (400, 401, 404):
            print(f"HTTP {http_status}")
            if exit_if_error:
                raise RuntimeError(f"HTTP error {http_status}")
            return None

    finally:
        resp.close()

    return out["data"]

# ========= 1. 用 token 登录，获取 apiKey =========
print("Logging in with token...")

login_payload = {
    "username": USERNAME,
    "token": TOKEN,         # 注意这里是 token 字段，不是 password
}

# 登录时一定要 api_key=None，避免带旧的、过期的 X-Auth-Token
api_key = send_request(SERVICE_URL + "login-token", login_payload, api_key=None)
print("API Key:", api_key)

# ========= 2. 构造空间 & 时间 & 云量过滤 =========
spatial_filter = {
    "filterType": "mbr",
    "lowerLeft":  {"latitude": LAT_MIN, "longitude": LON_MIN},
    "upperRight": {"latitude": LAT_MAX, "longitude": LON_MAX},
}

temporal_filter = {"start": START_DATE, "end": END_DATE}
cloud_cover_filter = {"min": 0, "max": 50}   # 示例：云量 0–50%

search_payload = {
    "datasetName": DATASET_NAME,
    "maxResults":  1,          # 只要一景，示例用
    "startingNumber": 1,
    "sceneFilter": {
        "spatialFilter":     spatial_filter,
        "acquisitionFilter": temporal_filter,
        "cloudCoverFilter":  cloud_cover_filter,
    },
    # 可选：按云量排序，最小云量优先
    "sortField": "cloudCover",
    "sortDirection": "ASC",
}

print("\nSearching scenes...")
scenes = send_request(SERVICE_URL + "scene-search", search_payload, api_key=api_key)

if not scenes["results"]:
    raise RuntimeError("No scenes found for given AOI/time range")

first_scene = scenes["results"][0]
entity_id   = first_scene["entityId"]
display_id  = first_scene.get("displayId", "<unknown>")
acq_date    = first_scene.get("acquisitionDate", "")

print(f"Found scene: {display_id} (entityId={entity_id}, date={acq_date})")

# ========= 3. 获取下载选项（download-options） =========
options_payload = {
    "datasetName": DATASET_NAME,
    "entityIds": [entity_id],
}

print("\nGetting download options...")
options = send_request(SERVICE_URL + "download-options", options_payload, api_key=api_key)

# 只保留可下载的选项（available == True）
available_opts = [opt for opt in options if opt.get("available")]
if not available_opts:
    raise RuntimeError("No available download options for this scene")

# 简单起见：用第一个可用产品（通常是整个打包 ZIP）
product_id = available_opts[0]["id"]
print("Using productId:", product_id, "productName:", available_opts[0].get("productName"))

downloads = [{"entityId": entity_id, "productId": product_id}]

# ========= 4. 发送 download-request =========
label = datetime.datetime.now().strftime("demo_%Y%m%d_%H%M%S")
request_payload = {
    "downloads": downloads,
    "label": label,
    "returnAvailable": True,   # 让已经准备好的下载立刻返回
}

print("\nRequesting download...")
req_result = send_request(SERVICE_URL + "download-request", request_payload, api_key=api_key)

preparing_ids = [d["downloadId"] for d in req_result.get("preparingDownloads", [])]
available     = req_result.get("availableDownloads", [])

# 先看立即可用的 URL
download_urls = [item["url"] for item in available]

# 如果还有没准备好的，再 poll 一次（简单示例，只 poll 一次）
if preparing_ids:
    print(f"{len(preparing_ids)} downloads still preparing, waiting 30s...")
    time.sleep(30)
    retrieve_payload = {"label": label}
    retrieve_result = send_request(
        SERVICE_URL + "download-retrieve",
        retrieve_payload,
        api_key=api_key,
        exit_if_error=False,
    )
    if retrieve_result:
        for item in retrieve_result.get("available", []):
            if item["downloadId"] in preparing_ids:
                download_urls.append(item["url"])

if not download_urls:
    raise RuntimeError("No download URLs returned")

download_url = download_urls[0]
print("\nFinal download URL:", download_url)

# ========= 5. 真正下载文件到本地 =========
out_path = OUTPUT_DIR / f"{display_id}.zip"
print(f"\nDownloading to {out_path} ...")

with requests.get(download_url, stream=True) as r:
    r.raise_for_status()
    with open(out_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

print("Done! Saved:", out_path)

# ========= 6. 登出（可选） =========
send_request(SERVICE_URL + "logout", {}, api_key=api_key)
print("Logged out.")


Logging in with token...
API Key: eyJjaWQiOjI3NzgyNDk1LCJzIjoiMTc2NTIxNzA2NSIsInIiOjQzNCwicCI6WyJ1c2VyIiwiZG93bmxvYWQiLCJvcmRlciJdfQ==

Searching scenes...
Found scene: LC08_L2SP_032039_20220608_20220616_02_T1 (entityId=LC80320392022159LGN00, date=)

Getting download options...
Using productId: 5e83d14fec7cae84 productName: Landsat Collection 2 Level-2 Product Bundle

Requesting download...

Final download URL: https://landsatlook.usgs.gov/gen-bundle?landsat_product_id=LC08_L2SP_032039_20220608_20220616_02_T1&requestSignature=eyJkb3dubG9hZEFwcCI6Ik0yTSIsImNvbnRhY3RJZCI6Mjc3ODI0OTUsImRvd25sb2FkSWQiOjkwNTk5MzY3NCwiZGF0ZUdlbmVyYXRlZCI6IjIwMjUtMTItMDhUMTI6MDQ6MjctMDY6MDAiLCJpZCI6IkxDMDhfTDJTUF8wMzIwMzlfMjAyMjA2MDhfMjAyMjA2MTZfMDJfVDEiLCJzaWduYXR1cmUiOiIkNSQkQjVkeWlWMUtpNHdKa0I4V2wzSFVObklJUVZBSk91Y2RvSjFUYXlpYUh5XC8ifQ==

Done! Saved: landsat_downloads/LC08_L2SP_032039_20220608_20220616_02_T1.zip
Logged out.


In [2]:
from pathlib import Path
import requests
from planetary_computer import sign
import pystac_client

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1"
)

items = catalog.search(
    collections=["landsat-c2-l2"],
    bbox=[-120, 30, -110, 40],
    datetime="2019-07-15/2019-08-03",
    query={"platform": {"in": ["landsat-8", "landsat-9"]}}
).item_collection()

print("Found items:", len(items))
if not items:
    raise SystemExit("没有找到数据，检查 bbox / 时间")

item = items[0]
print("Scene ID:", item.id)
print("Assets:", item.assets.keys())

outdir = Path("downloads")
outdir.mkdir(exist_ok=True)

def download_asset(key: str, suffix: str | None = None):
    asset = item.assets[key]
    url = sign(asset.href)
    name = f"{item.id}_{key}"
    if suffix:
        name = f"{item.id}_{suffix}"
    outpath = outdir / name
    print("Downloading", key, "->", outpath)

    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(outpath, "wb") as f:
            for chunk in r.iter_content(8192):
                f.write(chunk)
    return outpath

# 例子：下载 B4(red), B5(nir08), 热红外 lwir11, 以及 mtl.txt
download_asset("coastal")          # ≈ 1
download_asset("blue")         # ≈ SR_B2
download_asset("green")        # ≈ SR_B3
download_asset("red")          # ≈ SR_B4
download_asset("nir08")        # ≈ SR_B5
download_asset("swir16")       # ≈ SR_B6
download_asset("swir22")       # ≈ SR_B7
download_asset("lwir11")       # ≈ ST_B10 / 热红外
download_asset("mtl.txt", "MTL.txt")

print("Done.")


Found items: 72
Scene ID: LC08_L2SP_041038_20190802_02_T2
Assets: dict_keys(['qa', 'ang', 'red', 'blue', 'drad', 'emis', 'emsd', 'trad', 'urad', 'atran', 'cdist', 'green', 'nir08', 'lwir11', 'swir16', 'swir22', 'coastal', 'mtl.txt', 'mtl.xml', 'mtl.json', 'qa_pixel', 'qa_radsat', 'qa_aerosol', 'tilejson', 'rendered_preview'])


KeyboardInterrupt: 

In [5]:
print(item.assets.keys())


dict_keys(['qa', 'ang', 'red', 'blue', 'drad', 'emis', 'emsd', 'trad', 'urad', 'atran', 'cdist', 'green', 'nir08', 'lwir11', 'swir16', 'swir22', 'coastal', 'mtl.txt', 'mtl.xml', 'mtl.json', 'qa_pixel', 'qa_radsat', 'qa_aerosol', 'tilejson', 'rendered_preview'])


In [2]:
pip install planetary_computer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 19.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [planetary_computer]ydantic]
Note: you may need to restart the kernel to use updated packages.
